In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
# All imports consolidated here for the block synchronization pipeline

from __future__ import annotations
from pathlib import Path
from typing import Optional, Tuple, Literal, Sequence, Union
from dataclasses import dataclass
import re
import pickle

# Core data science
import numpy as np
import pandas as pd

# Bokeh for interactive visualization
from bokeh.io import output_notebook, show, reset_output
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CustomJS, Span, HoverTool
from bokeh.palettes import Category10
try:
    # Bokeh 3.x
    from bokeh.models import Slider
    from bokeh.resources import INLINE
except Exception:
    # Bokeh 2.x
    from bokeh.models.widgets import Slider
    from bokeh.resources import INLINE
from bokeh.layouts import column, row

# Project utilities
from eye_tracking_system_tools.preprocessing import utility_functions as uf

# Optional visualization (for jitter analysis plots)
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# simple approach
# --- Simple eye synchronization to the Open Ephys (OE) timebase ---
# Algorithm:
#   (1) Read internal timestamps (seconds) for each eye; delta-analysis to verify stability.
#   (2) Get the FIRST TTL sample for that eye from oe_events.
#   (3) Place frame 0 at that TTL, and place frame i at:  first_TTL_sample + round(fs * (t_sec[i] - t_sec[0])).
#       (i.e., use internal timing deltas verbatim; no regression.)
#   (4) Attach brightness from block.<le/re>_frame_val_list.
#   (5) Return a tidy per-eye DataFrame indexed by OE samples (int64), with OE time in seconds and brightness.



def _get_fs(block) -> float:
    """Return Open Ephys sample rate (Hz)."""
    fs = getattr(block, 'sample_rate', None)
    if fs is None:
        fs = float(block.get_sample_rate())
        block.sample_rate = fs
    return float(fs)

def _locate_eye_timestamps_csv(mp4: Path) -> Path:
    """
    Find <stem>_timestamps.csv even if the mp4 is *_LE.mp4 / *_RE.mp4
    while the CSV is <base>_timestamps.csv.
    """
    stem = mp4.stem
    stripped = re.sub(r'([_\-]?)(LE|RE)$', '', stem, flags=re.IGNORECASE)
    # 1) exact in same folder
    for s in (stem, stripped):
        p = mp4.with_name(s + "_timestamps.csv")
        if p.exists():
            return p
    # 2) fuzzy in same folder
    for pat in (f"{stripped}*timestamp*.csv", f"{stripped}*time*.csv",
                "*timestamp*.csv", "*time*.csv"):
        for p in mp4.parent.glob(pat):
            return p
    # 3) common subfolders
    for sub in ("timestamps", "frames_timestamps"):
        for d in (mp4.parent / sub, mp4.parent.parent / sub):
            if d.exists():
                for pat in (f"{stripped}*timestamp*.csv", f"{stripped}*time*.csv",
                            "*timestamp*.csv", "*time*.csv"):
                    for p in d.glob(pat):
                        return p
    # 4) LE/RE root recursive
    root = mp4.parents[1]
    for pat in (f"{stripped}*timestamp*.csv", f"{stripped}*time*.csv",
                "*timestamp*.csv", "*time*.csv"):
        for p in root.rglob(pat):
            return p
    raise FileNotFoundError(f"Timestamp CSV not found near {mp4}")

def _normalize_to_seconds(t: np.ndarray) -> np.ndarray:
    t = np.asarray(t, dtype=np.float64)
    dt = np.median(np.diff(t))

    # dt expected ~1/60 = 0.0167 s
    # If dt is ~16.7 -> ms
    # If dt is ~16666 -> us
    # If dt is ~1.666e7 -> ns
    if 0.005 < dt < 0.05:
        scale = 1.0          # already seconds
    elif 5 < dt < 50:
        scale = 1e-3         # ms -> s
    elif 5e3 < dt < 5e4:
        scale = 1e-6         # us -> s
    elif 5e6 < dt < 5e7:
        scale = 1e-9         # ns -> s
    else:
        # last resort: if values look like epoch seconds (1e9 range), keep but re-zero
        scale = 1.0

    t_s = t * scale
    # Re-zero to start at 0 to avoid epoch offsets
    t_s = t_s - t_s[0]
    return t_s

def _read_eye_internal_seconds(block, eye: str) -> np.ndarray:
    """
    Load per-frame internal timestamps for 'left' or 'right' eye and return as SECONDS.

    Robust to timestamp CSVs that store time in seconds / ms / us / ns (and to epoch-like offsets),
    by calling your pre-defined _normalize_to_seconds(t_raw).

    Requirements:
      - You already defined _normalize_to_seconds(t_raw: np.ndarray) -> np.ndarray
        which returns a strictly increasing array in seconds, re-zeroed at the first sample.

    Notes:
      - We try hard to pick a reasonable time column:
          1) prefer columns whose name suggests time (and prefer ones suggesting seconds)
          2) otherwise use the first numeric column
          3) last resort: read with header=None and take column 0
      - We validate monotonicity after normalization.
    """
    import re
    import numpy as np
    import pandas as pd
    from pathlib import Path

    # Ensure eye videos are known
    if getattr(block, 'le_videos', None) is None or getattr(block, 're_videos', None) is None:
        block.handle_eye_videos()

    mp4 = Path(block.le_videos[0] if eye == 'left' else block.re_videos[0])
    csvp = _locate_eye_timestamps_csv(mp4)

    # Read CSV (header expected, but tolerate odd formats)
    df = pd.read_csv(csvp, engine="python")

    def _is_numeric_col(s: pd.Series) -> bool:
        return pd.api.types.is_numeric_dtype(s) or pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s)

    # ---- Choose candidate time columns ----
    cols = list(df.columns)

    # Score columns: higher = better
    def _score_col(cname: str) -> int:
        c = str(cname).strip().lower()
        score = 0

        # must look like time-ish
        if "time" in c or "timestamp" in c or c in ("t", "ts"):
            score += 10
        if "frame" in c and ("time" in c or "timestamp" in c):
            score += 3

        # prefer explicit seconds
        if re.search(r"(\bsec\b|\bsecs\b|\bsecond\b|\bseconds\b|_s\b|\bs\b)", c):
            score += 6
        # de-prefer ms/us/ns if seconds also exist in file
        if re.search(r"(\bms\b|_ms\b|\bmsec\b)", c):
            score += 2
        if re.search(r"(\bus\b|_us\b|\busec\b)", c):
            score += 1
        if re.search(r"(\bns\b|_ns\b|\bnsec\b)", c):
            score += 0

        return score

    # Filter to numeric columns
    numeric_cols = [c for c in cols if _is_numeric_col(df[c])]

    # If we have time-ish numeric columns, pick the best scored among them
    timeish_numeric_cols = [c for c in numeric_cols
                            if ("time" in str(c).lower()) or ("timestamp" in str(c).lower()) or str(c).lower() in ("t", "ts")]
    chosen_col = None

    if timeish_numeric_cols:
        chosen_col = sorted(timeish_numeric_cols, key=_score_col, reverse=True)[0]
    elif numeric_cols:
        # fallback: first numeric column
        chosen_col = numeric_cols[0]
    else:
        # last resort: no header / no usable dtypes
        df2 = pd.read_csv(csvp, header=None, engine="python")
        chosen_col = 0
        df = df2

    # Extract raw time vector
    t_raw = df[chosen_col].to_numpy(dtype='float64')

    # Basic validation pre-normalization
    if t_raw.size < 3 or np.any(~np.isfinite(t_raw)):
        raise ValueError(f"Bad or too-short timestamps in {csvp} (column={chosen_col}).")

    # Normalize units to seconds (your helper)
    t_sec = _normalize_to_seconds(t_raw)

    # Validate output
    if t_sec.size < 3 or np.any(~np.isfinite(t_sec)):
        raise ValueError(f"Normalized timestamps invalid in {csvp} (column={chosen_col}).")
    if not np.all(np.diff(t_sec) > 0):
        # Give a more informative error (how many non-increasing steps)
        d = np.diff(t_sec)
        n_bad = int(np.sum(d <= 0))
        raise ValueError(
            f"Timestamps not strictly increasing after normalization in {csvp} "
            f"(column={chosen_col}); non-increasing steps: {n_bad}."
        )

    return t_sec


def _delta_analysis(t_sec: np.ndarray, label: str, cov_warn: float = 0.05) -> dict:
    """
    Basic delta analysis: fps median, CoV, outlier rate.
    Prints a short report; returns metrics.
    """
    dt = np.diff(t_sec)
    fps = 1.0 / np.median(dt)
    cov = float(np.std(dt) / np.mean(dt)) if np.mean(dt) > 0 else np.inf
    p01, p99 = np.percentile(dt, [1, 99])
    out_frac = float(np.mean((dt < p01) | (dt > p99)))
    print(f"[{label}] frames={len(t_sec):,} | median fps={fps:.3f} | CoV(dt)={cov*100:.2f}% | outliers(±1–99%)={out_frac*100:.2f}%")
    if cov > cov_warn:
        print(f"[WARN] {label}: CoV(dt) > {cov_warn*100:.1f}%. Stream may be unstable.")
    return dict(fps=fps, cov=cov, out_frac=out_frac, dt=dt)

def _first_ttl_sample(block, eye: str) -> int:
    """Get the FIRST TTL (OE samples) for the given eye."""
    col = 'L_eye_TTL' if eye == 'left' else 'R_eye_TTL'
    s = block.oe_events[col].dropna().astype(int).to_numpy()
    if s.size == 0:
        raise RuntimeError(f"No TTLs found for {eye} eye in oe_events['{col}'].")
    return int(s[0])

def build_eye_df_simple(block, eye: str, cov_warn: float = 0.05) -> pd.DataFrame:
    """
    Make a per-eye DataFrame indexed by OE samples using the simple anchor-at-first-TTL approach.
    Columns: ['frame_idx', 'oe_time_s', 'brightness'].
    """
    fs = _get_fs(block)
    t_sec = _read_eye_internal_seconds(block, eye)
    
    _delta_analysis(t_sec, label=eye.upper(), cov_warn=cov_warn)

    # anchor: first TTL sample; place frame 0 at this time; others by internal deltas
    t0_oe = _first_ttl_sample(block, eye)            # samples
    t_rel = t_sec - t_sec[0]                         # seconds relative to first frame
    oe_samples = t0_oe + np.round(fs * t_rel).astype(np.int64)

    # brightness from BlockSync
    b_list = getattr(block, 'le_frame_val_list' if eye == 'left' else 're_frame_val_list')
    b = np.asarray(b_list, dtype='float64')
    n = min(len(b), len(oe_samples))
    if len(b) != len(oe_samples):
        print(f"[INFO] {eye.upper()}: brightness length ({len(b)}) != frames ({len(oe_samples)}); clipping to {n}.")
    oe_samples = oe_samples[:n]
    b = b[:n]

    # Construct DataFrame (deduplicate OE stamps if rounding collided)
    df = pd.DataFrame({'frame_idx': np.arange(n, dtype=int),
                       'oe_sample': oe_samples,
                       'brightness': b})
    # If any duplicate oe_sample due to rounding, keep the first
    df = df.sort_values('oe_sample').drop_duplicates('oe_sample', keep='first')
    df['oe_time_s'] = df['oe_sample'] / fs
    df = df.set_index('oe_sample')
    return df

def simple_sync_build(block, cov_warn: float = 0.05, export: bool = False):
    """
    Run the simple synchronization for both eyes and (optionally) export CSVs.
    Returns (df_left, df_right).
    """
    dfL = build_eye_df_simple(block, 'left', cov_warn=cov_warn)
    dfR = build_eye_df_simple(block, 'right', cov_warn=cov_warn)
    if export:
        outL = Path(block.analysis_path) / "eye_left_simple_sync.csv"
        outR = Path(block.analysis_path) / "eye_right_simple_sync.csv"
        dfL.to_csv(outL); dfR.to_csv(outR)
        print(f"[OK] Saved: {outL}")
        print(f"[OK] Saved: {outR}")
    return dfL, dfR

# ============================================================================
# FUNCTION DEFINITIONS - Core Synchronization Functions
# ============================================================================
# All synchronization and utility functions consolidated here

# Note: plot_simple_sync_bokeh is defined in Cell 3 (better version with browser output)

def _get_fs(block) -> float:
    fs = getattr(block, 'sample_rate', None)
    if fs is None:
        fs = float(block.get_sample_rate()); block.sample_rate = fs
    return float(fs)

def _assert_strictly_increasing(name: str, arr: np.ndarray):
    if arr.size < 2 or not np.all(np.diff(arr) > 0):
        raise ValueError(f"{name} must be strictly increasing. Found non-monotonic sequence.")

def _build_arena_grid(block, target_fps: float):
    fs = _get_fs(block)

    arena = block.oe_events[['Arena_TTL','Arena_TTL_frame']].copy()
    arena = arena.dropna(subset=['Arena_TTL'])          # <- key change
    arena['Arena_TTL'] = arena['Arena_TTL'].astype(np.int64)
    arena = arena.sort_values('Arena_TTL')

    if len(arena) < 2:
        raise RuntimeError("Not enough Arena_TTL events to build a grid.")

    step  = int(round(fs / target_fps))
    start = int(arena['Arena_TTL'].iloc[0])
    stop  = int(arena['Arena_TTL'].iloc[-1])
    grid  = np.arange(start, stop + 1, step, dtype=np.int64)
    return fs, grid, step, arena


def _nearest_with_tol(sorted_vec: np.ndarray, queries: np.ndarray, tol: int) -> np.ndarray:
    """
    Return indices into sorted_vec of the nearest element to each query,
    but mark as -1 if the nearest is farther than tol (in samples).
    Assumes sorted_vec is strictly increasing (we assert that).
    """
    _assert_strictly_increasing("sorted_vec", sorted_vec)
    pos = np.searchsorted(sorted_vec, queries, side='left')
    pos0 = np.clip(pos - 1, 0, len(sorted_vec) - 1)
    pos1 = np.clip(pos,     0, len(sorted_vec) - 1)
    d0 = np.abs(sorted_vec[pos0] - queries)
    d1 = np.abs(sorted_vec[pos1] - queries)
    idx = np.where(d0 <= d1, pos0, pos1)
    d = np.minimum(d0, d1)
    idx[d > tol] = -1
    return idx

def _shift_eye_df_by_index(df: pd.DataFrame, shift: int) -> pd.DataFrame:
    """
    EXACT slider semantics on an eye df (index=oe_sample; cols: frame_idx, brightness, oe_time_s):
      - time index (oe_sample) and oe_time_s unchanged
      - shift BOTH frame_idx and brightness by `shift` along the current order
      - edges filled with NaN, no wrap
    """
    if shift == 0:
        return df.copy()
    df = df.copy()
    n = len(df)
    fi = df['frame_idx'].to_numpy(dtype=float)
    br = df['brightness'].to_numpy(dtype=float)
    fi_sh = np.full(n, np.nan, dtype=float)
    br_sh = np.full(n, np.nan, dtype=float)
    if shift > 0:
        fi_sh[shift:] = fi[:-shift]
        br_sh[shift:] = br[:-shift]
    else:
        s = -int(shift)
        fi_sh[:-s] = fi[s:]
        br_sh[:-s] = br[s:]
    df['frame_idx']  = fi_sh
    df['brightness'] = br_sh
    return df

def describe_eye_tick(df: pd.DataFrame) -> float:
    """
    Return the median sampling interval in milliseconds for this eye dataframe.
    df must have an 'oe_time_s' column (from simple_sync_build).
    """
    t = df['oe_time_s'].to_numpy(dtype=float)
    if len(t) < 2:
        return float('nan')
    return float(np.median(np.diff(t)) * 1000.0)

def shift_eye_df_by_index(df: pd.DataFrame, shift: int) -> pd.DataFrame:
    """
    Apply the SAME 'index-based shift' as the Bokeh slider to an eye dataframe:
      - Keep time ('oe_time_s') and the OE-sample index (df.index) unchanged.
      - Shift BOTH 'frame_idx' and 'brightness' by `shift` along the time-sorted order.
      - Fill the vacated edges with NaN (no wrap-around).
    Assumes df is the output of simple_sync_build (index = oe_sample, cols include 'frame_idx','brightness','oe_time_s').
    """
    if shift == 0:
        return df.copy()

    # Ensure sorted by time (simple_sync_build already gives it, but be safe)
    df = df.sort_index().copy()

    # Build new columns by discrete shift
    n = len(df)
    frame_idx = df['frame_idx'].to_numpy(dtype=float)   # float to allow NaN
    bright    = df['brightness'].to_numpy(dtype=float)

    shifted_idx = np.full(n, np.nan, dtype=float)
    shifted_y   = np.full(n, np.nan, dtype=float)

    if shift > 0:
        shifted_idx[shift:] = frame_idx[:-shift]
        shifted_y[shift:]   = bright[:-shift]
    else:
        s = -int(shift)
        shifted_idx[:-s] = frame_idx[s:]
        shifted_y[:-s]   = bright[s:]

    out = df.copy()
    out['frame_idx'] = shifted_idx
    out['brightness'] = shifted_y
    return out

def build_final_sync_df_merge_nearest(
    block,
    df_left:  Optional[pd.DataFrame],
    df_right: Optional[pd.DataFrame],
    target_fps: float = 60.0,
    tol_frac: float = 0.9,          # nearest is accepted if within tol_frac * tick
    pre_shift_left:  int = 0,       # apply EXACT slider-like index shift BEFORE merge
    pre_shift_right: int = 0,
    export_csv: bool = True,
    csv_name: str = "blocksync_df.csv",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Build downstream-compatible final_sync_df by merging df_left/df_right onto a 60 Hz Arena grid
    using nearest-with-tolerance, without re-sorting the eye dataframes.

    Expects df_left/df_right from simple_sync_build (index=oe_sample; cols: frame_idx, oe_time_s, brightness).
    """
    # 1) Build Arena grid
    fs, grid, step, arena = _build_arena_grid(block, target_fps=target_fps)
    tol = int(np.ceil(tol_frac * step))  # in samples
    tick_ms = 1000.0 * step / fs

    # 2) Eye dfs: sanity + optional pre-merge slider-like shifts (NO sorting is performed)
    for nm, df in (("LEFT", df_left), ("RIGHT", df_right)):
        if not isinstance(df.index.values, np.ndarray):
            raise ValueError(f"{nm}: df.index must be OE samples.")
        _assert_strictly_increasing(f"{nm} df.index (oe_sample)", df.index.values.astype(np.int64))
        if not {'frame_idx','brightness','oe_time_s'}.issubset(df.columns):
            raise ValueError(f"{nm}: df must include 'frame_idx','brightness','oe_time_s'.")

    if pre_shift_left:
        if verbose: print(f"[INFO] Pre-shifting LEFT by {pre_shift_left} ticks (slider semantics).")
        df_left = _shift_eye_df_by_index(df_left, pre_shift_left)
    if pre_shift_right:
        if verbose: print(f"[INFO] Pre-shifting RIGHT by {pre_shift_right} ticks (slider semantics).")
        df_right = _shift_eye_df_by_index(df_right, pre_shift_right)

    # 3) Map Arena frames to the grid (nearest-with-tolerance)
    a_times  = arena['Arena_TTL'].to_numpy(dtype=np.int64)
    a_frames = arena['Arena_TTL_frame'].to_numpy(dtype=np.int64)
    idxA = _nearest_with_tol(a_times, grid, tol)
    arena_frame = np.full(grid.shape, np.nan, dtype=float)
    okA = idxA >= 0
    arena_frame[okA] = a_frames[idxA[okA]]

    # 4) Nearest-with-tolerance merge for LEFT and RIGHT (NO resort of dfs)
    def _map_eye_to_grid(df_eye: pd.DataFrame):
        t   = df_eye.index.to_numpy(dtype=np.int64)       # oe_sample per frame
        fi  = df_eye['frame_idx'].to_numpy(dtype=float)
        val = df_eye['brightness'].to_numpy(dtype=float)
        idx = _nearest_with_tol(t, grid, tol)             # indices into t
        frames = np.full(grid.shape, np.nan, dtype=float)
        vals   = np.full(grid.shape, np.nan, dtype=float)
        ok = idx >= 0
        frames[ok] = fi[idx[ok]]
        vals[ok]   = val[idx[ok]]
        return frames, vals

    L_eye_frame, L_values = _map_eye_to_grid(df_left)
    R_eye_frame, R_values = _map_eye_to_grid(df_right)

    # 5) Assemble final df (identical column names/format to legacy)
    final_df = pd.DataFrame({
        'Arena_TTL':   grid.astype(float),   # float to mimic legacy style (… .0)
        'Arena_frame': arena_frame,
        'L_eye_frame': L_eye_frame,
        'R_eye_frame': R_eye_frame,
        'L_values':    L_values,
        'R_values':    R_values,
    })

    # 6) Save & attach
    if export_csv:
        outp = Path(block.analysis_path) / csv_name
        final_df.to_csv(outp, index=False)
        if verbose: print(f"[OK] Saved final sync CSV → {outp}")
    block.final_sync_df = final_df

    if verbose:
        print(f"[INFO] Grid rows: {len(grid):,} | tick ≈ {tick_ms:.3f} ms | tol={tol} samp (tol_frac={tol_frac:.2f})")
        nL = int(np.sum(np.isfinite(L_values))); nR = int(np.sum(np.isfinite(R_values)))
        print(f"[INFO] Valid LEFT grid points: {nL:,} | Valid RIGHT grid points: {nR:,}")

    return final_df

def sanity_plot_final_df(final_df, fs, show_led_off=False, led_off_samples=None, title="final_df sanity"):
    """
    final_df: DataFrame with columns ['Arena_TTL','L_values','R_values']
    fs:       Open Ephys sample rate (Hz)
    show_led_off: if True, draw vertical lines at LED-OFF sample indices in led_off_samples
    """
    x_s  = np.asarray(final_df['Arena_TTL'], dtype=float) / fs
    yL   = np.asarray(final_df['L_values'], dtype=float)
    yR   = np.asarray(final_df['R_values'], dtype=float)

    output_notebook()
    p = figure(title=title, x_axis_label="OE time (s)", y_axis_label="Brightness (a.u.)",
               width=1200, height=450, tools="pan,wheel_zoom,box_zoom,reset,save")
    src = ColumnDataSource(dict(x=x_s, yL=yL, yR=yR))
    p.line('x', 'yL', source=src, line_width=1.5, color="#1f77b4", legend_label="Left eye")
    p.line('x', 'yR', source=src, line_width=1.5, color="#d62728", legend_label="Right eye")
    p.legend.click_policy = "hide"

    if show_led_off and led_off_samples is not None and len(led_off_samples):
        for s in led_off_samples:
            p.add_layout(Span(location=float(s)/fs, dimension='height',
                              line_color="#2ca02c", line_alpha=0.5, line_width=1.5))
    show(p)



@dataclass
class ArenaGridInfo:
    fs_hz: float
    inferred_arena_fps: float
    inferred_arena_step_samp: int
    target_fps: float
    used_pseudo_60hz: bool
    start_samp: int
    end_samp: int
    n_grid: int


def _infer_ttl_fps(samples: np.ndarray, fs_hz: float) -> Tuple[float, int]:
    """
    Robust estimate of TTL cadence: inferred_fps = fs / median(diff(samples)).
    Returns (fps, median_step_samples).
    """
    samples = np.asarray(samples, dtype=np.int64)
    samples = np.sort(samples)
    if samples.size < 3:
        return float("nan"), -1
    dt = np.diff(samples)
    # ignore zeros/negatives just in case (shouldn't happen)
    dt = dt[dt > 0]
    if dt.size < 2:
        return float("nan"), -1
    step = int(np.median(dt))
    fps = float(fs_hz / step) if step > 0 else float("nan")
    return fps, step


def build_arena_grid_df(
    block,
    target_fps: float = 60.0,
    arena_fps_tol_hz: float = 5.0,
    arena_ttl_col: str = "Arena_TTL",
    arena_frame_col: str = "Arena_TTL_frame",
    prefer_valid_window: bool = True,
    verbose: bool = True,
    attach_to_block: bool = True,
) -> Tuple[pd.DataFrame, ArenaGridInfo]:
    """
    Create an arena 'master grid' DataFrame on the Open Ephys sample axis.

    If inferred arena fps is within ±arena_fps_tol_hz of target_fps:
        - use the arena TTL-derived grid (legacy paradigm)
        - the returned df represents the arena's native stream (≈60Hz)
    Else (arena slower/faster than expected):
        - create a pseudo 60Hz grid from arena-valid time window
        - upsample arena frames onto it by nearest assignment (duplicates frames)

    Returns
    -------
    arena_grid_df : DataFrame indexed by OE samples (int64)
        Columns:
          - 'grid_sample' (index duplicate)
          - 'grid_time_s'
          - 'Arena_frame_60'   : 0..N-1 (frame index on the 60Hz grid)
          - 'Arena_frame_src'  : mapped original arena frame number (duplicates allowed)
          - 'Arena_ttl_src'    : OE sample timestamp of the source arena TTL nearest each grid tick
    info : ArenaGridInfo
    """

    if getattr(block, "oe_events", None) is None:
        raise RuntimeError("block.oe_events is missing. Run block.parse_open_ephys_events() first.")
    fs = float(getattr(block, "sample_rate", None) or block.get_sample_rate())

    oe = block.oe_events

    if arena_ttl_col not in oe.columns:
        raise RuntimeError(f"'{arena_ttl_col}' not found in block.oe_events columns.")

    # Use TTL timestamps that exist; optionally restrict to the "valid window" as defined by your parser cleaning
    arena = oe[[arena_ttl_col] + ([arena_frame_col] if arena_frame_col in oe.columns else [])].copy()
    arena = arena.dropna(subset=[arena_ttl_col])
    arena[arena_ttl_col] = arena[arena_ttl_col].astype(np.int64)
    arena = arena.sort_values(arena_ttl_col)

    if len(arena) < 3:
        raise RuntimeError("Not enough arena TTL events to infer fps / build grid.")

    # If parser set arena_frame to NaN outside window, we can use that to find the valid window
    if prefer_valid_window and (arena_frame_col in arena.columns):
        # keep only TTLs that have a non-NaN frame counter (i.e., in-window after your cleaning)
        arena_in = arena.dropna(subset=[arena_frame_col]).copy()
        if len(arena_in) >= 3:
            arena_use = arena_in
        else:
            arena_use = arena
    else:
        arena_use = arena

    a_samp = arena_use[arena_ttl_col].to_numpy(dtype=np.int64)
    fps_a, step_a = _infer_ttl_fps(a_samp, fs)

    if verbose:
        print(f"[arena] inferred fps ≈ {fps_a:.3f} Hz (median step {step_a} samples @ fs={fs:.1f} Hz)")

    # Define the usable sync window for building the grid
    start_samp = int(a_samp[0])
    end_samp   = int(a_samp[-1])

    # Decide: native vs pseudo
    use_native = np.isfinite(fps_a) and (abs(fps_a - target_fps) <= arena_fps_tol_hz)

    if use_native:
        # Native-like grid: based on target_fps step, but spanning native window.
        # (If fps_a is ~60, this matches your legacy expectation.)
        step_grid = int(round(fs / float(target_fps)))
        grid = np.arange(start_samp, end_samp + 1, step_grid, dtype=np.int64)
        used_pseudo = False
    else:
        # Pseudo 60Hz grid spanning the arena window
        step_grid = int(round(fs / float(target_fps)))
        grid = np.arange(start_samp, end_samp + 1, step_grid, dtype=np.int64)
        used_pseudo = True
        if verbose:
            print(f"[arena] arena fps not ~{target_fps}. Building pseudo {target_fps}Hz grid over window.")

    # Map each grid tick to a source arena TTL (nearest)
    # This is the key step that duplicates arena frames when arena is slower (e.g., 15Hz)
    a_samp_full = arena_use[arena_ttl_col].to_numpy(dtype=np.int64)
    a_frame_full = None
    if arena_frame_col in arena_use.columns:
        a_frame_full = arena_use[arena_frame_col].to_numpy(dtype=float)  # may have NaNs
    else:
        # if no frame col, create one as 0..n-1 for the arena TTL list
        a_frame_full = np.arange(len(arena_use), dtype=float)

    # nearest mapping via searchsorted
    pos = np.searchsorted(a_samp_full, grid, side="left")
    pos0 = np.clip(pos - 1, 0, len(a_samp_full) - 1)
    pos1 = np.clip(pos,     0, len(a_samp_full) - 1)
    d0 = np.abs(a_samp_full[pos0] - grid)
    d1 = np.abs(a_samp_full[pos1] - grid)
    idx = np.where(d0 <= d1, pos0, pos1)

    arena_ttl_src = a_samp_full[idx]
    arena_frame_src = a_frame_full[idx]

    arena_grid_df = pd.DataFrame({
        "grid_sample": grid.astype(np.int64),
        "grid_time_s": grid.astype(np.float64) / fs,
        "Arena_frame_60": np.arange(len(grid), dtype=np.int64),  # the 60Hz grid frame index
        "Arena_frame_src": arena_frame_src,                      # original arena frame index (duplicates allowed)
        "Arena_ttl_src": arena_ttl_src.astype(np.int64),
    }).set_index("grid_sample")

    info = ArenaGridInfo(
        fs_hz=fs,
        inferred_arena_fps=float(fps_a),
        inferred_arena_step_samp=int(step_a),
        target_fps=float(target_fps),
        used_pseudo_60hz=bool(used_pseudo),
        start_samp=int(start_samp),
        end_samp=int(end_samp),
        n_grid=int(len(grid)),
    )

    if attach_to_block:
        block.arena_grid_df = arena_grid_df
        block.arena_grid_info = info

    if verbose:
        src_unique = int(pd.Series(arena_frame_src).nunique(dropna=True))
        print(f"[arena] grid rows={len(arena_grid_df):,} | unique source arena frames mapped={src_unique:,}")
        if used_pseudo:
            print("[arena] NOTE: Arena_frame_src will repeat (upsampled arena). Use Arena_frame_60 as the 60Hz master frame index.")

    return arena_grid_df, info


# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def plot_simple_sync_bokeh(block, df_left=None, df_right=None, shift_range=200, show_led=True, to_browser=True):
    """
    Bokeh plot of both eyes' brightness vs Open Ephys time (seconds), with manual shift sliders.
    Opens in the system default browser using a temporary HTML (not saved in your project).
    """
    # Helper to get fs
    def _get_fs(b) -> float:
        fs = getattr(b, 'sample_rate', None)
        if fs is None:
            fs = float(b.get_sample_rate()); b.sample_rate = fs
        return float(fs)

    # Build simple sync if needed
    if df_left is None or df_right is None:
        df_left, df_right = simple_sync_build(block, export=False)

    fs = _get_fs(block)

    # Convert NaNs/Infs in Y to None for Bokeh
    def _nan2none(a):
        return [None if (not np.isfinite(v)) else float(v) for v in a]

    xL = df_left['oe_time_s'].to_numpy(dtype=float)
    yL = df_left['brightness'].to_numpy(dtype=float)
    xR = df_right['oe_time_s'].to_numpy(dtype=float)
    yR = df_right['brightness'].to_numpy(dtype=float)

    stepL_ms = float(np.median(np.diff(xL))*1000.0) if len(xL) > 1 else float('nan')
    stepR_ms = float(np.median(np.diff(xR))*1000.0) if len(xR) > 1 else float('nan')
    print(f"[INFO] Slider tick ≈ {stepL_ms:.3f} ms (Left), {stepR_ms:.3f} ms (Right)")

    src_le = ColumnDataSource(dict(x=xL.tolist(), y=_nan2none(yL), y0=_nan2none(yL)))
    src_re = ColumnDataSource(dict(x=xR.tolist(), y=_nan2none(yR), y0=_nan2none(yR)))

    # Figure
    p = figure(title="Simple synchronization — brightness vs OE time (s)  (zoom/pan; use sliders to shift)",
               x_axis_label="OE time (s)", y_axis_label="Brightness (a.u.)",
               width=1200, height=450, tools="pan,wheel_zoom,box_zoom,reset,save")

    p.line('x', 'y', source=src_le, line_width=1.5, color="#1f77b4", legend_label="Left eye")
    p.line('x', 'y', source=src_re, line_width=1.5, color="#d62728", legend_label="Right eye")
    p.legend.click_policy = "hide"

    # LED verticals
    if show_led and ('LED_driver' in getattr(block, 'oe_events', {}).columns):
        led = block.oe_events['LED_driver'].dropna().astype(int).to_numpy()
        if led.size:
            led_s = led / fs
            for x in led_s:
                p.add_layout(Span(location=float(x), dimension='height',
                                  line_color="#2ca02c", line_alpha=0.5, line_width=1.5))

    # Sliders
    sL = Slider(title="Left Eye Shift (indices)",  start=-shift_range, end=shift_range, value=0, step=1, width=350)
    sR = Slider(title="Right Eye Shift (indices)", start=-shift_range, end=shift_range, value=0, step=1, width=350)

    cb = CustomJS(args=dict(le=src_le, re=src_re), code="""
        const sL = sL_slider.value|0;
        const sR = sR_slider.value|0;

        // LEFT
        const yL  = le.data['y'];
        const yL0 = le.data['y0'];
        const NL  = yL.length;
        for (let i=0; i<NL; i++) {
            const j = i + sL;
            yL[i] = (j>=0 && j<NL) ? yL0[j] : null;
        }
        le.change.emit();

        // RIGHT
        const yR  = re.data['y'];
        const yR0 = re.data['y0'];
        const NR  = yR.length;
        for (let i=0; i<NR; i++) {
            const j = i + sR;
            yR[i] = (j>=0 && j<NR) ? yR0[j] : null;
        }
        re.change.emit();
    """)
    cb.args['sL_slider'] = sL
    cb.args['sR_slider'] = sR
    sL.js_on_change('value', cb)
    sR.js_on_change('value', cb)

    layout = column(p, row(sL, sR))

    # Open in default browser without saving to your project
    reset_output()
    if to_browser:
        show(layout)
    else:
        output_notebook(resources=INLINE, hide_banner=True)
        show(layout)


def hover_inspect_eyes_bokeh(df_left, df_right, title="Hover inspect — per-eye stream on OE time"):
    """
    Hover over each trace to see oe_sample (index), oe_time_s, frame_idx, brightness.
    Works on df outputs of simple_sync_build() or shifted versions.
    """
    output_notebook()

    def _prep(df, label):
        d = df.sort_index().copy()
        return ColumnDataSource(dict(
            oe_time_s=d["oe_time_s"].to_numpy(dtype=float),
            brightness=d["brightness"].to_numpy(dtype=float),
            frame_idx=d["frame_idx"].to_numpy(dtype=float),
            oe_sample=d.index.to_numpy(dtype=np.int64),
            label=np.array([label]*len(d), dtype=object),
        ))

    srcL = _prep(df_left,  "LEFT")
    srcR = _prep(df_right, "RIGHT")

    p = figure(
        title=title,
        x_axis_label="OE time (s)",
        y_axis_label="Brightness (a.u.)",
        width=1200,
        height=450,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    rL = p.line("oe_time_s", "brightness", source=srcL, line_width=1.5, color="#1f77b4", legend_label="Left eye")
    rR = p.line("oe_time_s", "brightness", source=srcR, line_width=1.5, color="#d62728", legend_label="Right eye")
    p.legend.click_policy = "hide"

    # Separate hover tools
    htL = HoverTool(
        renderers=[rL],
        mode="vline",
        tooltips=[
            ("eye", "@label"),
            ("oe_time_s", "@oe_time_s{0.000}"),
            ("oe_sample", "@oe_sample{0}"),
            ("frame_idx", "@frame_idx{0}"),
            ("brightness", "@brightness{0.00}"),
        ],
    )
    htR = HoverTool(
        renderers=[rR],
        mode="vline",
        tooltips=[
            ("eye", "@label"),
            ("oe_time_s", "@oe_time_s{0.000}"),
            ("oe_sample", "@oe_sample{0}"),
            ("frame_idx", "@frame_idx{0}"),
            ("brightness", "@brightness{0.00}"),
        ],
    )
    p.add_tools(htL, htR)
    show(column(p))


# ============================================================================
# FRAME INSERTION FUNCTIONS (for dropped frame correction)
# ============================================================================

InsertMode = Literal["prev", "current"]

def _normalize_insert_positions(
    df: pd.DataFrame,
    insert_at: Sequence[Union[int, np.integer]],
    *,
    mode: Literal["pos", "oe_sample"] = "pos",
) -> np.ndarray:
    """
    Convert user-provided insert points into 0..N-1 row positions in df's sorted-by-index order.
    mode="pos": insert_at already refers to row positions
    mode="oe_sample": insert_at are OE samples (df.index values) -> map to nearest row position
    """
    n = len(df)
    if n == 0:
        return np.array([], dtype=int)

    insert_at = np.asarray(insert_at, dtype=np.int64)

    if mode == "pos":
        pos = np.clip(insert_at, 0, n - 1)
        return pos.astype(int)

    if mode == "oe_sample":
        idx = df.sort_index().index.to_numpy(dtype=np.int64)
        p = np.searchsorted(idx, insert_at, side="left")
        p0 = np.clip(p - 1, 0, n - 1)
        p1 = np.clip(p,     0, n - 1)
        d0 = np.abs(idx[p0] - insert_at)
        d1 = np.abs(idx[p1] - insert_at)
        pos = np.where(d0 <= d1, p0, p1)
        return pos.astype(int)

    raise ValueError("mode must be 'pos' or 'oe_sample'.")


def insert_duplicate_frames_slide(
    df: pd.DataFrame,
    insert_at: Sequence[Union[int, np.integer]],
    *,
    mode: Literal["pos", "oe_sample"] = "pos",
    duplicate: InsertMode = "prev",
    cols: Sequence[str] = ("frame_idx", "brightness"),
    leave_trailing_nan: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Insert a duplicated frame at one or more points, sliding CONTENT forward by 1 tick each time,
    while keeping the OE time axis fixed (df.index and oe_time_s unchanged).
    """
    if len(df) == 0:
        return df.copy()

    d = df.sort_index().copy()
    n = len(d)

    pos = _normalize_insert_positions(d, insert_at, mode=mode)
    pos = np.sort(pos)

    if verbose:
        print(f"[INFO] Will insert {len(pos)} duplicated frame(s) at positions: {pos.tolist()} (mode={mode}, duplicate={duplicate})")

    out = d.copy()
    for c in cols:
        if c not in out.columns:
            raise ValueError(f"Column '{c}' not in df.")
        out[c] = out[c].to_numpy(dtype=float)

    for k in pos:
        for c in cols:
            arr = out[c].to_numpy(dtype=float)

            if duplicate == "prev":
                src_val = arr[k - 1] if k > 0 else np.nan
            else:
                src_val = arr[k]

            arr[k + 1:] = arr[k:-1]
            arr[k] = src_val

            if leave_trailing_nan:
                arr[-1] = np.nan

            out[c] = arr

    return out


def insert_dup_by_pos(df, positions, **kwargs):
    """Convenience wrapper for inserting by position."""
    return insert_duplicate_frames_slide(df, positions, mode="pos", **kwargs)


def insert_dup_by_oe_sample(df, oe_samples, **kwargs):
    """Convenience wrapper for inserting by OE sample."""
    return insert_duplicate_frames_slide(df, oe_samples, mode="oe_sample", **kwargs)


# ============================================================================
# VERIFICATION AND EXPORT FUNCTIONS
# ============================================================================

def verify_final_df_against_sources(block, final_df, df_left, df_right, target_fps=60.0, tol_frac=0.9):
    """Verify that final_df matches recomputed mapping from source eye dataframes."""
    fs, grid, step, arena = _build_arena_grid(block, target_fps)
    tol = int(np.ceil(tol_frac * step))

    ft = np.asarray(final_df['Arena_TTL'], dtype=float)

    if len(ft) != len(grid):
        print("[VERIFY] Length mismatch:")
        print("  len(final_df Arena_TTL) =", len(ft))
        print("  len(recomputed grid)    =", len(grid))
        raise AssertionError("Grid length mismatch.")

    if not np.allclose(ft, grid.astype(float), rtol=0, atol=0.5):
        bad = np.where(np.abs(ft - grid.astype(float)) > 0.5)[0][:10]
        print("[VERIFY] Arena_TTL mismatch at indices:", bad)
        raise AssertionError("final_df['Arena_TTL'] does not match recomputed grid.")

    def _nearest_with_tol(sorted_vec, queries, tol):
        _assert_strictly_increasing("sorted_vec", sorted_vec)
        pos = np.searchsorted(sorted_vec, queries, 'left')
        pos0 = np.clip(pos-1, 0, len(sorted_vec)-1)
        pos1 = np.clip(pos,   0, len(sorted_vec)-1)
        d0 = np.abs(sorted_vec[pos0] - queries)
        d1 = np.abs(sorted_vec[pos1] - queries)
        idx = np.where(d0 <= d1, pos0, pos1)
        d = np.minimum(d0, d1)
        idx[d > tol] = -1
        return idx

    def map_eye(df_eye):
        t = df_eye.sort_index().index.to_numpy(dtype=np.int64)
        _assert_strictly_increasing("df_eye.index (oe_sample)", t)
        fi = df_eye.sort_index()['frame_idx'].to_numpy(dtype=float)
        y  = df_eye.sort_index()['brightness'].to_numpy(dtype=float)
        idx = _nearest_with_tol(t, grid, tol)
        frames = np.full(grid.shape, np.nan, dtype=float)
        vals   = np.full(grid.shape, np.nan, dtype=float)
        ok = idx >= 0
        frames[ok] = fi[idx[ok]]
        vals[ok]   = y[idx[ok]]
        return frames, vals

    Lf, Lv = map_eye(df_left)
    Rf, Rv = map_eye(df_right)

    Lf0 = final_df['L_eye_frame'].to_numpy(dtype=float)
    Lv0 = final_df['L_values'].to_numpy(dtype=float)
    Rf0 = final_df['R_eye_frame'].to_numpy(dtype=float)
    Rv0 = final_df['R_values'].to_numpy(dtype=float)

    def nan_equal(a,b):
        return ((a==b) | (np.isnan(a) & np.isnan(b)))

    mLf = nan_equal(Lf, Lf0).mean()
    mLv = nan_equal(Lv, Lv0).mean()
    mRf = nan_equal(Rf, Rf0).mean()
    mRv = nan_equal(Rv, Rv0).mean()

    n = len(grid)
    print(f"[VERIFY] Grid length={n}, tick≈{1000.0*step/fs:.3f} ms, tol={tol} samples")
    print(f"[VERIFY] Left  frame match: {mLf*100:.3f}%   Left  values match: {mLv*100:.3f}%")
    print(f"[VERIFY] Right frame match: {mRf*100:.3f}%   Right values match: {mRv*100:.3f}%")

    return dict(
        tick_ms=1000.0*step/fs, tol_samples=tol,
        left_frame_match=mLf, left_values_match=mLv,
        right_frame_match=mRf, right_values_match=mRv
    )


def export_final_sync_df(block,
                         final_df: pd.DataFrame,
                         overwrite: bool = True,
                         filenames = ("blocksync_df.csv", "final_sync_df.csv"),
                         ms_axis=True) -> None:
    """
    Save `final_df` into block.analysis_path and set block.final_sync_df and block.blocksync_df.
    """
    required = ['Arena_TTL','Arena_frame','L_eye_frame','R_eye_frame','L_values','R_values']
    missing = [c for c in required if c not in final_df.columns]
    if missing:
        raise ValueError(f"final_df missing required columns: {missing}")

    out = final_df.copy()
    out['Arena_TTL'] = out['Arena_TTL'].astype(float)
    if ms_axis:
        out['ms_axis'] = out['Arena_TTL'] / (block.sample_rate/1000)
    
    ap = Path(block.analysis_path)
    ap.mkdir(parents=True, exist_ok=True)

    for name in filenames:
        p = ap / name
        existed = p.exists()
        if existed and not overwrite:
            print(f"[SKIP] {p.name} exists and overwrite=False")
            continue
        out.to_csv(p, index=False)
        print(f"[OK] {'Overwrote' if existed else 'Wrote'} {p}")

    block.final_sync_df = out
    block.blocksync_df = out
    print("[OK] block.final_sync_df (and block.blocksync_df) set.")


def load_final_sync_df(block, filename=None, verbose=True):
    """
    Load a downstream-compatible final sync dataframe from disk and set block.final_sync_df and block.blocksync_df.
    """
    ap = Path(block.analysis_path)
    candidates = [filename] if filename else ["final_sync_df.csv", "blocksync_df.csv"]
    path = None
    for name in candidates:
        p = ap / name
        if p.exists():
            path = p
            break
    if path is None:
        raise FileNotFoundError(f"No sync file found. Tried: {', '.join(str(ap / n) for n in candidates)}")

    df = pd.read_csv(path)
    required = ['Arena_TTL','Arena_frame','L_eye_frame','R_eye_frame','L_values','R_values']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    df = df.copy()
    df['Arena_TTL'] = df['Arena_TTL'].astype(float)

    block.final_sync_df = df
    block.blocksync_df = df
    if verbose:
        print(f"[OK] Loaded {path.name} → block.final_sync_df (rows={len(df):,})")

    return df


# ============================================================================
# JITTER ANALYSIS FUNCTIONS
# ============================================================================

def create_distance_plot(distances, top_dist_to_show=500):
    """Create cumulative distribution and histogram plots for jitter distances."""
    sns.set(style="whitegrid")
    fig, axs = plt.subplots(2, figsize=(6, 6), dpi=150)

    axs[0].set_title('Cumulative Euclidean Distances for Camera Jitter', fontsize=15)
    axs[0].set_ylabel('Cumulative \n % of Frames')
    axs[0].set_xlim(0, top_dist_to_show)
    axs[0].grid(False)

    sns.kdeplot(distances, cumulative=True, label='Left Eye', ax=axs[0], linewidth=4, c='black')

    axs[1].hist(distances, bins=np.linspace(0, top_dist_to_show, 20), log=False, color='black')
    axs[1].set_title('Image displacement histogram', fontsize=15)
    axs[1].set_xlabel('Euclidean Displacement [$\mu$m]', fontsize=15)
    axs[1].set_ylabel('Frame count', fontsize=15)
    axs[1].tick_params(axis='x', which='major', labelsize=15)
    axs[1].set_facecolor('white')
    axs[1].title.set_color('black')
    axs[1].xaxis.label.set_color('black')
    axs[1].yaxis.label.set_color('black')
    axs[1].tick_params(colors='black')
    axs[1].grid(False)

    plt.tight_layout()
    return fig, axs


def add_intermediate_elements(input_vector, gap_to_bridge):
    """Add intervening elements to bridge gaps in a vector."""
    differences = np.diff(input_vector)
    output_vector = [input_vector[0]]
    for i, diff in enumerate(differences):
        if diff < gap_to_bridge:
            output_vector.extend(range(input_vector[i] + 1, input_vector[i + 1]))
        output_vector.append(input_vector[i + 1])
    return np.sort(np.unique(output_vector))


def find_jittery_frames(block, eye, max_distance, diff_threshold, gap_to_bridge=6):
    """Find jittery frames based on distance thresholds and return indices to remove."""
    if eye not in ['left', 'right']:
        print(f'eye can only be left/right, your input: {eye}')
        return None
    
    if eye == 'left':
        jitter_dict = block.le_jitter_dict
        eye_frame_col = 'L_eye_frame'
    elif eye == 'right':
        jitter_dict = block.re_jitter_dict
        eye_frame_col = 'R_eye_frame'

    df_dict = {'left':block.le_df, 'right':block.re_df}
    df = pd.DataFrame.from_dict(jitter_dict)
    
    indices_of_highest_drift = df.query("top_correlation_dist > @max_distance").index.values
    diff_vec = np.diff(df['top_correlation_dist'].values)
    diff_peaks_indices = np.where(diff_vec > diff_threshold)[0]
    video_indices = np.concatenate((diff_peaks_indices, indices_of_highest_drift))
    print(f'the diff based jitter frame exclusion gives: {np.shape(diff_peaks_indices)}')
    print(f'the threshold based jitter frame exclusion gives: {np.shape(indices_of_highest_drift)}')

    video_indices = add_intermediate_elements(video_indices, gap_to_bridge=gap_to_bridge)
    df_indices_to_remove = df_dict[eye].loc[df_dict[eye][eye_frame_col].isin(video_indices)].index.values

    return df_indices_to_remove, video_indices


# ============================================================================
# FINAL EXPORT FUNCTIONS
# ============================================================================

def export_eye_data_2d(block):
    """
    Save the eye dataframes to two CSV files.
    :param block: The current blocksync class with verified re/le dfs
    :return: None
    """
    block.right_eye_data.to_csv(block.analysis_path / 'right_eye_data.csv')
    block.left_eye_data.to_csv(block.analysis_path / 'left_eye_data.csv')
    print(f'eye dataframes saved to: {block.analysis_path}')


# Block Synchronization Pipeline

This notebook implements the complete block synchronization workflow, bringing eye tracking data from raw videos to synchronized `left_eye_data` and `right_eye_data` DataFrames ready for downstream analysis.

## Workflow Overview

1. **Setup**: Initialize BlockSync object
2. **Data Preparation**: Handle videos, parse Open Ephys events, extract brightness
3. **Arena Grid**: Build 60Hz master grid from arena TTL events
4. **Simple Synchronization**: Initial eye sync using first TTL anchor
5. **Manual Correction**: Interactive alignment with LED events
6. **Final Merge**: Merge corrected eyes onto arena grid
7. **Verification**: Check alignment quality
8. **Jitter Correction**: Remove camera jitter artifacts
9. **LED Blink Removal**: Remove LED blink artifacts
10. **Final Export**: Create left/right_eye_data for downstream use

---

## Step 1: Block Setup and Initialization

In [ ]:
# block instantiation:
bad_blocks = [] #
experiment_path = Path(r"D:\sample_data_for_eye_repo")

block_numbers = [15]
animal = 'PV_106'
block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks,regev=True,
                                      )
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
    elif block.animal_call == "TE_21":
        block.channeldict={1:'Arena_TTL',
                           4:'LED_driver',
                           5:'R_eye_TTL',
                           8:'L_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b

## Step 2: Data Preparation

Run the following methods to prepare the block data:
- `handle_eye_videos()`: Convert and validate eye video files
- `parse_open_ephys_events()`: Parse Open Ephys events (includes manual TTL selector for non-standard paradigms)
- `handle_arena_files()`: Process arena video files
- `get_eye_brightness_vectors()`: Extract brightness values from eye videos

## Step 3: Build Arena Grid

Create the master 60Hz grid from arena TTL events. This grid will be used to align all streams. The function automatically handles cases where arena fps differs from 60Hz by creating a pseudo-60Hz grid.

In [ ]:
block = block_collection[0]
block.handle_eye_videos()
block.parse_open_ephys_events()
block.handle_arena_files()
block.get_eye_brightness_vectors()


## Step 4: Simple Eye Synchronization

Build per-eye DataFrames using the simple anchor-at-first-TTL approach. This creates initial synchronization that can be manually corrected in the next step.

The algorithm:
1. Read internal timestamps for each eye
2. Get the FIRST TTL sample for that eye
3. Place frame 0 at that TTL, others by internal timing deltas
4. Attach brightness values

In [ ]:
# after block.parse_open_ephys_events() succeeded (manual or auto)
arena_grid_df, info = build_arena_grid_df(block, target_fps=60.0, arena_fps_tol_hz=5.0)

# info.used_pseudo_60hz tells you whether it applied the correction
print(info)

## Step 5: Manual Synchronization Correction

Use the interactive Bokeh plot to visually align the eye traces with LED events. Adjust sliders to shift traces, then apply corrections using `shift_eye_df_by_index()`.

**Note**: The plot opens in your default browser. Use the sliders to find the correct shift values, then apply them in the next cell.

In [ ]:
# Build and save the per-eye DataFrames
dfL, dfR = simple_sync_build(block, export=True)

# Quick look and manual correction to LED grid
plot_simple_sync_bokeh(block, dfL, dfR, show_led=True)


In [ ]:
# Inspect ms per tick (helps translate slider “ticks” to ms)
print("Left tick ≈ %.3f ms"  % describe_eye_tick(dfL))
print("Right tick ≈ %.3f ms" % describe_eye_tick(dfR))

## Step 6: Merge onto Arena Grid

Merge the corrected eye DataFrames onto the 60Hz arena grid using nearest-neighbor matching with tolerance. This creates the final synchronized dataframe (`final_sync_df`) used downstream.

In [ ]:
# Use the function to apply shift, 
# Notice: the values here should be inverse to the shift which corrects the interactive plot
dfL_shifted = shift_eye_df_by_index(dfL,-3)
dfR_shifted = shift_eye_df_by_index(dfR,9)

## Step 7: Verification

Verify the final synchronization by plotting and checking alignment statistics.

In [ ]:
# use this function to verify shift implemented
plot_simple_sync_bokeh(block, dfL_shifted, dfR_shifted, show_led=True)

In [ ]:
# Optional: Hover inspection plot for detailed frame-by-frame inspection
# hover_inspect_eyes_bokeh(dfL_shifted, dfR_shifted)


In [ ]:
# Optional: Insert duplicate frames to correct for dropped frames
# Example usage (uncomment if needed):
# dfL_fix = insert_dup_by_pos(dfL_shifted, [3], duplicate="prev", leave_trailing_nan=True)
# dfR_fix = insert_dup_by_pos(dfR_shifted, [3], duplicate="prev", leave_trailing_nan=True)



In [ ]:
# Merge onto the 60 Hz arena grid with nearest-with-tolerance (no resort of eye dfs)
final_df = build_final_sync_df_merge_nearest(
    block, dfL_shifted, dfR_shifted,
    target_fps=60.0,
    tol_frac=0.90,      # accept nearest within 90% of a 60 Hz tick; tune 0.7–1.2 if needed
    pre_shift_left=0,   # IMPORTANT: already pre-shifted
    pre_shift_right=0,
    export_csv=True
)

# 3) Sanity check: this should now reproduce the same crisp alignment you saw pre-merge
#sanity_check_final_sync_bokeh(block, final_df=final_df, show_led=True, shift_range=200)

In [ ]:

fs = float(block.sample_rate)
sanity_plot_final_df(final_df, fs, show_led_off=True, led_off_samples=block.oe_events['LED_driver'].dropna().astype(int).to_numpy())


## Step 8: Jitter Correction and LED Blink Removal

Correct for camera jitter and remove LED blink artifacts from the eye dataframes. This step:
1. Generates jitter reports
2. Corrects jitter using cross-correlation
3. Identifies and removes LED blink frames
4. Removes jittery frames based on distance thresholds

In [ ]:
#debugging here:
fs = float(block.sample_rate)

# 1) Compare end-times (in seconds)
arena_ttl = block.oe_events['Arena_TTL'].dropna().astype(np.int64)
print("Arena TTL span (s):", (arena_ttl.iloc[-1] - arena_ttl.iloc[0]) / fs)

print("Grid span (s):", (final_df['Arena_TTL'].iloc[-1] - final_df['Arena_TTL'].iloc[0]) / fs)

print("Left eye span (s):", (dfL_shifted.index.max() - dfL_shifted.index.min()) / fs)
print("Right eye span (s):", (dfR_shifted.index.max() - dfR_shifted.index.min()) / fs)

# 2) Where do values stop being finite?
lastL = final_df['L_values'].last_valid_index()
lastR = final_df['R_values'].last_valid_index()
print("Last valid L row:", lastL, "time(s):", final_df['Arena_TTL'].iloc[lastL] / fs if lastL is not None else None)
print("Last valid R row:", lastR, "time(s):", final_df['Arena_TTL'].iloc[lastR] / fs if lastR is not None else None)


In [ ]:
# dfL_s/dfR_s are the eye DFs you actually shifted (exact slider semantics)
stats = verify_final_df_against_sources(block, final_df, dfL_shifted, dfR_shifted, target_fps=60.0, tol_frac=0.9)


In [ ]:
# Export the final synchronized dataframe
export_final_sync_df(block, final_df=final_df, overwrite=True)

### preprocessing of synchronized data from here (loads final_sync from folder if available)

In [ ]:

for block in block_collection:
    load_final_sync_df(block)


In [ ]:
for block in block_collection:
    # load relevant data
    block.handle_eye_videos()
    block.parse_open_ephys_events()
    block.handle_arena_files()
    block.get_eye_brightness_vectors()
    load_final_sync_df(block)
    # read deeplabcut annotations and construct ellipse parameters per video frame
    block.read_dlc_data(overwrite=False, export=True)

In [ ]:
for block in block_collection:
    # cross-corr distance estimation between eye video frames to correct for jitter
    # (relatively slow, but only needs to be run once per block, will load from disk if available)
    block.get_jitter_reports(export=True, overwrite=False, remove_led_blinks=False, sort_on_loading=True)

In [ ]:
# perform jitter correction and remove led blinks
for block in block_collection:
    block.correct_jitter()
    block.find_led_blink_frames(plot=True)
    block.remove_led_blinks_from_eye_df(export=True)

In [ ]:
df_inds_to_remove_l, vid_inds_l = find_jittery_frames(block, 'left', max_distance=60, diff_threshold=5,
                                                      gap_to_bridge=24)
df_inds_to_remove_r, vid_inds_r = find_jittery_frames(block, 'right', max_distance=60, diff_threshold=5,
                                                      gap_to_bridge=24)

# These are verification plots for the jitter outlier removal functions:
# to verify, I want a bokeh explorable:
rdf = pd.DataFrame.from_dict(block.re_jitter_dict)
ldf = pd.DataFrame.from_dict(block.le_jitter_dict)

In [ ]:
# visualize right eye
uf.bokeh_plotter([rdf.top_correlation_dist], ['drift_distance'], peaks=vid_inds_r)

In [ ]:
# visualize left eye
uf.bokeh_plotter([ldf.top_correlation_dist], ['drift_distance'], peaks=vid_inds_l)

In [ ]:
# if you are happy with the results, remove the outliers
block.remove_eye_datapoints_based_on_video_frames('right', indices_to_nan=vid_inds_r)
block.remove_eye_datapoints_based_on_video_frames('left', indices_to_nan=vid_inds_l)

In [ ]:
# create the eye dataframes, integrating the cleaned le/re dataframes with the ellipse parameters
# this will create block.left_eye_data and block.right_eye_data
for block in block_collection:
    block.create_eye_data()

In [ ]:

# Export the final eye dataframes
for block in block_collection:
    export_eye_data_2d(block)